In [19]:
import requests
import json
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import FastEmbedEmbeddings


def scrape_page(url):
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    paragraphs = [p.get_text() for p in soup.find_all("p")]
    code_blocks = [code.get_text() for code in soup.find_all("code")]

    return {
        "text": "\n".join(paragraphs),
        "code": "\n\n".join(code_blocks)
    }


# Scrape multiple pages for better coverage
urls = [
    "https://cadquery.readthedocs.io/en/latest/apireference.html"
]

docs = []
for url in urls:
    try:
        data = scrape_page(url)
        docs.append({
            "source": url,
            "content": data["text"],
            "code": data["code"]
        })
        print(f"Scraped: {url}")
    except Exception as e:
        print(f"Failed to scrape {url}: {e}")

    

Scraped: https://cadquery.readthedocs.io/en/latest/apireference.html


In [23]:
combined = data['text'] + "\n" + data['code']
combined

'The CadQuery API is made up of 4 main objects:\nSketch – Construct 2D sketches\nWorkplane – Wraps a topological entity and provides a 2D modelling context.\nSelector – Filter and select things\nAssembly – Combine objects into assemblies.\nThis page lists  methods of these objects grouped by functional area\nSee also\nThis page lists api methods grouped by functional area.\nUse CadQuery Class Summary to see methods alphabetically by class.\nCreating new sketches.\nSketch(parent,\xa0locs,\xa0obj)\n2D sketch.\nSketch.importDXF(filename[,\xa0tol,\xa0exclude,\xa0...])\nImport a DXF file and construct face(s)\nWorkplane.sketch()\nInitialize and return a sketch\nSketch.finalize()\nFinish sketch construction and return the parent.\nSketch.copy()\nCreate a partial copy of the sketch.\nSketch.located(loc)\nCreate a partial copy of the sketch with a new location.\nSketch.moved(...)\nCreate a partial copy of the sketch with moved _faces.\nSelecting, tagging and manipulating elements.\nSketch.tag(

In [22]:
print(type(data))
print(data.keys())
print(data)

<class 'dict'>
dict_keys(['text', 'code'])
{'text': 'The CadQuery API is made up of 4 main objects:\nSketch – Construct 2D sketches\nWorkplane – Wraps a topological entity and provides a 2D modelling context.\nSelector – Filter and select things\nAssembly – Combine objects into assemblies.\nThis page lists  methods of these objects grouped by functional area\nSee also\nThis page lists api methods grouped by functional area.\nUse CadQuery Class Summary to see methods alphabetically by class.\nCreating new sketches.\nSketch(parent,\xa0locs,\xa0obj)\n2D sketch.\nSketch.importDXF(filename[,\xa0tol,\xa0exclude,\xa0...])\nImport a DXF file and construct face(s)\nWorkplane.sketch()\nInitialize and return a sketch\nSketch.finalize()\nFinish sketch construction and return the parent.\nSketch.copy()\nCreate a partial copy of the sketch.\nSketch.located(loc)\nCreate a partial copy of the sketch with a new location.\nSketch.moved(...)\nCreate a partial copy of the sketch with moved _faces.\nSelect

In [30]:
SECTIONS = {
    "Workplane_3D": [
        "Workplane.extrude", "Workplane.box", "Workplane.sphere",
        "Workplane.cut", "Workplane.cutBlind", "Workplane.cutThruAll",
        "Workplane.loft", "Workplane.sweep", "Workplane.twistExtrude",
        "Workplane.revolve", "Workplane.shell", "Workplane.fillet",
        "Workplane.chamfer", "Workplane.split", "Workplane.rotate",
        "Workplane.rotateAboutCenter", "Workplane.translate", "Workplane.mirror",
        "Workplane.union", "Workplane.combine", "Workplane.intersect",
        "Workplane.wedge", "Workplane.cylinder", "Workplane.cboreHole",
        "Workplane.cskHole", "Workplane.hole", "Workplane.text",
    ]
}

def get_section(method_name):
    for section, keywords in SECTIONS.items():
        if any(method_name.startswith(k) for k in keywords):
            return section
    return None  # None means skip this method

def chunk_with_metadata(data, source_url):
    text_lines = data['text'].split('\n')
    code_lines = data['code'].split('\n')

    methods = [line.strip() for line in code_lines if line.strip()]

    current_methods = []
    current_texts = []

    for method in methods:
        section = get_section(method)
        if section is None:
            continue  # skip non-3D methods

        current_methods.append(method)
        matching = [t for t in text_lines if method.split('(')[0] in t]
        if matching:
            current_texts.extend(matching)

    chunk = {
        "content": "\n".join(current_texts),
        "metadata": {
            "source": source_url,
            "section": "Workplane_3D",
            "type": "api_reference",
            "library": "cadquery",
            "methods": current_methods,
            "method_count": len(current_methods),
        }
    }

    return [chunk]

# Run it
chunks = chunk_with_metadata(data, "https://cadquery.readthedocs.io/en/latest/apireference.html")

# Preview
for c in chunks:
    print(f"\n--- Section: {c['metadata']['section']} ({c['metadata']['method_count']} methods) ---")
    print(f"Methods: {c['metadata']['methods']}")
    print(f"\nContent:\n{c['content']}")


--- Section: Workplane_3D (30 methods) ---
Methods: ['Workplane.mirrorY', 'Workplane.mirrorX', 'Workplane.cboreHole', 'Workplane.cskHole', 'Workplane.hole', 'Workplane.extrude', 'Workplane.cut', 'Workplane.cutBlind', 'Workplane.cutThruAll', 'Workplane.box', 'Workplane.sphere', 'Workplane.wedge', 'Workplane.cylinder', 'Workplane.union', 'Workplane.combine', 'Workplane.intersect', 'Workplane.loft', 'Workplane.sweep', 'Workplane.twistExtrude', 'Workplane.revolve', 'Workplane.text', 'Workplane.shell', 'Workplane.fillet', 'Workplane.chamfer', 'Workplane.split', 'Workplane.rotate', 'Workplane.rotateAboutCenter', 'Workplane.translate', 'Workplane.mirror', 'Workplane.shells']

Content:
Workplane.mirrorY()
Workplane.mirrorX()
Workplane.cboreHole(diameter, cboreDiameter, ...)
Workplane.cskHole(diameter, cskDiameter, ...)
Workplane.hole(diameter[, depth, clean])
Workplane.extrude(until[, combine, clean, ...])
Workplane.cut(toCut[, clean, tol])
Workplane.cutBlind(until[, clean, both, taper])
Work

In [32]:
chunks[0]

{'content': 'Workplane.mirrorY()\nWorkplane.mirrorX()\nWorkplane.cboreHole(diameter,\xa0cboreDiameter,\xa0...)\nWorkplane.cskHole(diameter,\xa0cskDiameter,\xa0...)\nWorkplane.hole(diameter[,\xa0depth,\xa0clean])\nWorkplane.extrude(until[,\xa0combine,\xa0clean,\xa0...])\nWorkplane.cut(toCut[,\xa0clean,\xa0tol])\nWorkplane.cutBlind(until[,\xa0clean,\xa0both,\xa0taper])\nWorkplane.cutThruAll([clean,\xa0taper])\nWorkplane.cutBlind(until[,\xa0clean,\xa0both,\xa0taper])\nWorkplane.cutThruAll([clean,\xa0taper])\nWorkplane.box(length,\xa0width,\xa0height[,\xa0...])\nWorkplane.sphere(radius[,\xa0direct,\xa0angle1,\xa0...])\nWorkplane.wedge(dx,\xa0dy,\xa0dz,\xa0xmin,\xa0zmin,\xa0...)\nWorkplane.cylinder(height,\xa0radius,\xa0direct,\xa0...)\nWorkplane.union([toUnion,\xa0clean,\xa0glue,\xa0tol])\nWorkplane.combine([clean,\xa0glue,\xa0tol])\nWorkplane.intersect(toIntersect[,\xa0clean,\xa0tol])\nWorkplane.loft([ruled,\xa0combine,\xa0clean])\nWorkplane.sweep(path[,\xa0multisection,\xa0...])\nWorkpla